<a href="https://colab.research.google.com/github/Siya-Tambe/AlgoSaathi/blob/main/AlgoSaathi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import yfinance as yf #yfinance - Yahoo finance library is to get free stock data
import pandas as pd
def fetch_data(ticker="RELIANCE.NS",period="1y"): #  ticker - symbol for stock (.NSE means NSE listed)
  data = yf.download(ticker,period=period, progress=False)
  data = data[["Close"]].reset_index() #We will be considering only closing price
  data.columns = ["Date","Close"]
  return data

In [41]:
df=fetch_data(ticker="RELIANCE.NS",period="1y")

short_window = 20
long_window = 50

df.head()

/tmp/ipykernel_2509/3566533761.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker,period=period, progress=False)


,Date,Close
0,2025-08-18,1375.340942
1,2025-08-19,1413.564209
2,2025-08-20,1406.496948
3,2025-08-21,1418.242676
4,2025-08-22,1402.714355


In [42]:
df["SMA_short"] = df["Close"].rolling(window=short_window).mean()
df["SMA_long"] = df["Close"].rolling(window=long_window).mean()

In [43]:
df["Signal"]=0
df.loc[df["SMA_short"] > df["SMA_long"], "Signal"]=1
df["Position"]=df["Signal"].diff()

In [44]:
df["Daily_return"]=df["Close"].pct_change()
df["Strategy_return"]=df["Daily_return"]*df["Signal"].shift(1)

In [45]:
initial_capital = 100000
df["Strategy_Equity"] =  initial_capital * (1 + df["Strategy_return"]).cumprod()
df["Buy_Hold_Equity"]= initial_capital * (1 + df["Daily_return"]).cumprod()

In [46]:
def calculate_rsi(df, period=14):
  delta = df["Close"].diff()

  gain = delta.clip(lower=0)
  loss = -delta.clip(upper=0)

  avg_gain = gain.rolling(window=period).mean()
  avg_loss = loss.rolling(window=period).mean()

  rs = avg_gain / avg_loss
  df["RSI"]= 100 - (100/(1+rs))
  return df

In [47]:
df = calculate_rsi(df, period=14)
df[["Date", "Close", "RSI"]].tail(10)

,Date,Close,RSI
239,2026-08-04,1290.900024,49.443568
240,2026-08-05,1280.000000,46.067988
241,2026-08-06,1325.000000,55.892118
242,2026-08-07,1334.800049,51.725725
243,2026-08-10,1327.300049,50.939193
244,2026-08-11,1323.900024,54.865138
245,2026-08-12,1329.000000,65.673286
246,2026-08-13,1317.000000,60.405541
247,2026-08-14,1310.000000,57.796252
248,2026-08-17,1316.000000,62.976900


In [48]:
df["Signal"]=0
df.loc[
    (df["SMA_short"] > df["SMA_long"]) & (df["RSI"]<70),
     "Signal"
] = 1

In [51]:
print("Total BUY signal days: ", (df["Signal"] == 1).sum())
df["Position"] = df["Signal"].diff()

Total BUY signal days:  60
